In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()
RESULTS_ROOT = Path(os.environ.get("ANALYSIS_RESULTS_DIR", DATA_ROOT.parent / "analysis_code" / "results")).expanduser().resolve()


In [ ]:
# Mitochondrial genes were excluded from visualization, and the top-ranked remaining genes are shown.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Nature-style dot plot for top-ranked B-cell genes across major PBMC cell types.

Dot size:
    Fraction of cells expressing the gene, calculated from layers["counts"] > 0.

Dot color:
    Mean normalized expression, calculated from adata.X.

Features:
    1. Read ranked genes from gene_rank.csv.
    2. Replace AnnData var_names with gene symbols.
    3. Map detailed PBMC annotations to major cell types.
    4. Remove missing/unmapped cells.
    5. Optionally remove mitochondrial and ribosomal genes.
    6. Continue down the ranking until TOP_N valid genes are obtained.
    7. Export editable PDF/SVG and high-resolution PNG.
    8. Export all numerical data used for plotting.
"""

from pathlib import Path
import re
import sys
import warnings

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp


# ============================================================
# 1. Input and output paths
# ============================================================
GENE_RANK_FILE = DATA_ROOT / "Figure6/pbmc/gene_rank.csv"

RNA_H5AD_FILE = DATA_ROOT / "Figure6/pbmc/homo_sapiens_rna.h5ad"

OUTPUT_DIR = RESULTS_ROOT / "Figure6/pbmc/ranked_gene_expression"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Parameters
# ============================================================
CELLTYPE_COLUMN = "Celltype"
GENE_SYMBOL_COLUMN = "gene_symbol"
PLOT_CELLTYPE_COLUMN = "major_celltype"

# Final number of genes displayed
TOP_N = 10

# Remove mitochondrial genes such as MT-CO1 and MT-CO2
FILTER_MT_GENES = True

# Optionally remove ribosomal protein genes such as RPL/RPS
FILTER_RIBOSOMAL_GENES = False

# Optional manually excluded genes
MANUAL_EXCLUDE_GENES = set()

# Remove genes that show zero expression in all retained cell types
REMOVE_ZERO_EXPRESSION_GENES = True

# Dot size
MIN_DOT_SIZE = 18
MAX_DOT_SIZE = 620

# Clip extreme mean-expression values for the color scale
COLOR_CLIP_PERCENTILE = 99

# Size legend fractions
SIZE_LEGEND_FRACTIONS = [
    0.10,
    0.30,
    0.50,
    0.75,
]

OUTPUT_PREFIX = (
    f"top{TOP_N}_ranked_genes_"
    "major_celltype_dotplot_nature"
)


# ============================================================
# 3. Detailed-to-major cell-type mapping
# ============================================================
CELLTYPE_TO_MAJOR = {
    # CD4 T cell
    "CD4 Naive": "CD4 T cell",
    "CD4 TCM": "CD4 T cell",
    "CD4 TEM": "CD4 T cell",
    "Treg": "CD4 T cell",

    # CD8 T cell
    "CD8 Naive": "CD8 T cell",
    "CD8 TEM_1": "CD8 T cell",
    "CD8 TEM_2": "CD8 T cell",

    # Innate-like T cell
    "MAIT": "Innate-like T cell",
    "gdT": "Innate-like T cell",

    # NK cell
    "NK": "NK cell",

    # Monocyte
    "CD14 Mono": "Monocyte",
    "CD16 Mono": "Monocyte",

    # Dendritic cell
    "cDC": "Dendritic cell",
    "pDC": "Dendritic cell",

    # B cell
    "Naive B": "B cell",
    "Memory B": "B cell",
    "Intermediate B": "B cell",

    # Plasma cell
    "Plasma": "Plasma cell",

    # HSPC
    "HSPC": "HSPC",
}


MAJOR_CELLTYPE_ORDER = [
    "B cell",
    "Plasma cell",
    "CD4 T cell",
    "CD8 T cell",
    "Innate-like T cell",
    "NK cell",
    "Monocyte",
    "Dendritic cell",
    "HSPC",
]


# ============================================================
# 4. Matplotlib settings
# ============================================================
mpl.rcParams.update(
    {
        # Editable text in vector outputs
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],

        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",

        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 11,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,

        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,

        "savefig.dpi": 600,
    }
)


# ============================================================
# 5. Custom continuous colormap
# ============================================================
NATURE_CMAP = LinearSegmentedColormap.from_list(
    "nature_expression",
    [
        "#F4F6F7",
        "#DCE8F2",
        "#AFCBE3",
        "#6CA6CD",
        "#358F9B",
        "#52A66D",
        "#A8C95F",
        "#E6C84F",
    ],
    N=256,
)


# ============================================================
# 6. Utility functions
# ============================================================
def require_file(path: Path) -> None:
    """Check that a file exists and is not empty."""
    if not path.is_file():
        raise FileNotFoundError(
            f"File not found: {path}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"File is empty: {path}"
        )


def normalize_gene_symbol(value) -> str:
    """Clean and uppercase a gene symbol."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value in {
        "",
        "nan",
        "None",
        "NA",
        "<NA>",
    }:
        return ""

    return value.upper()


def mean_axis0(matrix) -> np.ndarray:
    """Calculate column means from sparse or dense matrices."""
    if sp.issparse(matrix):
        return np.asarray(
            matrix.mean(axis=0)
        ).ravel()

    return np.asarray(matrix).mean(
        axis=0
    )


def fraction_positive_axis0(matrix) -> np.ndarray:
    """Calculate the fraction of values greater than zero."""
    if sp.issparse(matrix):
        return np.asarray(
            (matrix > 0).mean(axis=0)
        ).ravel()

    return (
        np.asarray(matrix) > 0
    ).mean(axis=0)


def combine_columns(
    matrix,
    column_indices,
):
    """
    Sum duplicate features representing the same gene symbol.
    """
    sub = matrix[:, column_indices]

    if len(column_indices) == 1:
        result = sub

    elif sp.issparse(sub):
        result = sp.csr_matrix(
            sub.sum(axis=1)
        )

    else:
        result = np.asarray(
            sub
        ).sum(
            axis=1,
            keepdims=True,
        )

    if sp.issparse(result):
        result = result.tocsr()

        if result.shape[1] != 1:
            result = sp.csr_matrix(
                np.asarray(result).reshape(-1, 1)
            )

    else:
        result = np.asarray(
            result
        ).reshape(-1, 1)

    return result


def classify_filtered_gene(
    gene_symbol: str,
):
    """
    Return a filtering reason or None.
    """
    symbol = normalize_gene_symbol(
        gene_symbol
    )

    if (
        FILTER_MT_GENES
        and symbol.startswith("MT-")
    ):
        return "mitochondrial_gene"

    if (
        FILTER_RIBOSOMAL_GENES
        and (
            symbol.startswith("RPL")
            or symbol.startswith("RPS")
        )
    ):
        return "ribosomal_gene"

    manual_excluded = {
        normalize_gene_symbol(x)
        for x in MANUAL_EXCLUDE_GENES
    }

    if symbol in manual_excluded:
        return "manual_exclusion"

    return None


# ============================================================
# 7. Read and filter ranking table
# ============================================================
def load_gene_rank(
    path: Path,
) -> pd.DataFrame:
    """
    Read gene_rank.csv and optionally filter mitochondrial,
    ribosomal, or manually excluded genes.
    """
    rank_df = pd.read_csv(
        path
    )

    required_columns = {
        "gene",
        "score",
        "gene_symbol",
        "gene_id",
    }

    missing = required_columns.difference(
        rank_df.columns
    )

    if missing:
        raise ValueError(
            "Missing required columns in gene rank file: "
            f"{sorted(missing)}"
        )

    rank_df["score"] = pd.to_numeric(
        rank_df["score"],
        errors="coerce",
    )

    rank_df["gene_symbol"] = (
        rank_df["gene_symbol"]
        .map(normalize_gene_symbol)
    )

    rank_df["gene_id"] = (
        rank_df["gene_id"]
        .astype(str)
        .str.strip()
    )

    rank_df = rank_df[
        rank_df["gene_symbol"].ne("")
        & rank_df["score"].notna()
    ].copy()

    rank_df = (
        rank_df
        .sort_values(
            "score",
            ascending=False,
        )
        .drop_duplicates(
            subset="gene_symbol",
            keep="first",
        )
        .reset_index(drop=True)
    )

    # Original rank before filtering
    rank_df["original_rank"] = (
        np.arange(len(rank_df)) + 1
    )

    rank_df["filter_reason"] = (
        rank_df["gene_symbol"]
        .map(classify_filtered_gene)
    )

    filtered_out = rank_df[
        rank_df["filter_reason"].notna()
    ].copy()

    retained = rank_df[
        rank_df["filter_reason"].isna()
    ].copy()

    # Rank after filtering
    retained["filtered_rank"] = (
        np.arange(len(retained)) + 1
    )

    filtered_out.to_csv(
        OUTPUT_DIR / "filtered_ranked_genes.csv",
        index=False,
    )

    retained.to_csv(
        OUTPUT_DIR / "retained_ranked_genes.csv",
        index=False,
    )

    print()
    print("=" * 72)
    print("Gene-ranking filtering")
    print("=" * 72)
    print(
        f"Original unique ranked genes: "
        f"{len(rank_df):,}"
    )
    print(
        f"Retained genes:              "
        f"{len(retained):,}"
    )
    print(
        f"Filtered genes:              "
        f"{len(filtered_out):,}"
    )

    print()
    print("Filter settings:")
    print(
        f"  FILTER_MT_GENES:         "
        f"{FILTER_MT_GENES}"
    )
    print(
        f"  FILTER_RIBOSOMAL_GENES:  "
        f"{FILTER_RIBOSOMAL_GENES}"
    )
    print(
        f"  MANUAL_EXCLUDE_GENES:    "
        f"{sorted(MANUAL_EXCLUDE_GENES)}"
    )

    if not filtered_out.empty:
        print()
        print("Top filtered genes:")
        print(
            filtered_out[
                [
                    "original_rank",
                    "gene_symbol",
                    "score",
                    "filter_reason",
                ]
            ]
            .head(20)
            .to_string(index=False)
        )

    print()
    print(
        f"Top {TOP_N} retained genes "
        "before RNA matching:"
    )
    print(
        retained[
            [
                "original_rank",
                "filtered_rank",
                "gene_symbol",
                "score",
            ]
        ]
        .head(TOP_N)
        .to_string(index=False)
    )

    return retained


# ============================================================
# 8. Prepare AnnData gene symbols
# ============================================================
def prepare_gene_symbols(
    adata: ad.AnnData,
) -> ad.AnnData:
    """
    Replace var_names with uppercase gene symbols.

    Features without valid symbols are removed.
    Duplicate symbols remain but are made unique.
    """
    if GENE_SYMBOL_COLUMN not in adata.var.columns:
        raise KeyError(
            f"{GENE_SYMBOL_COLUMN!r} "
            "is not present in adata.var."
        )

    symbols = (
        adata.var[GENE_SYMBOL_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    valid = (
        symbols.notna()
        & symbols.ne("")
        & symbols.ne("NAN")
        & symbols.ne("NONE")
        & symbols.ne("<NA>")
    )

    n_invalid = int(
        (~valid).sum()
    )

    if n_invalid > 0:
        print()
        print(
            f"Removing {n_invalid:,} features "
            "without valid gene symbols."
        )

        adata = adata[
            :,
            valid.to_numpy(),
        ].copy()

        symbols = symbols.loc[
            valid
        ].copy()

    adata.var["original_var_name"] = (
        adata.var_names.astype(str)
    )

    adata.var[
        "gene_symbol_original_case"
    ] = (
        adata.var[GENE_SYMBOL_COLUMN]
        .astype(str)
    )

    adata.var_names = (
        symbols.astype(str).to_numpy()
    )

    n_duplicates = int(
        adata.var_names.duplicated().sum()
    )

    if n_duplicates > 0:
        warnings.warn(
            f"Found {n_duplicates:,} duplicate gene-symbol "
            "features. AnnData suffixes will be added."
        )

        adata.var_names_make_unique(
            join="-"
        )

    # Scanpy compatibility
    adata.uns.setdefault(
        "log1p",
        {},
    )

    adata.uns["log1p"].setdefault(
        "base",
        None,
    )

    print()
    print("=" * 72)
    print("Gene-symbol preparation")
    print("=" * 72)
    print(
        f"Genes retained: "
        f"{adata.n_vars:,}"
    )
    print(
        f"Unique var_names: "
        f"{adata.var_names.is_unique}"
    )

    return adata


# ============================================================
# 9. Map cell types and remove unknown cells
# ============================================================
def prepare_celltypes(
    adata: ad.AnnData,
) -> ad.AnnData:
    """
    Map detailed annotations to major PBMC cell types.

    Missing and unmapped cells are removed.
    No Unknown category is retained.
    """
    if CELLTYPE_COLUMN not in adata.obs.columns:
        raise KeyError(
            f"{CELLTYPE_COLUMN!r} "
            "is not present in adata.obs."
        )

    original = (
        adata.obs[CELLTYPE_COLUMN]
        .astype("string")
        .str.strip()
    )

    major = original.map(
        CELLTYPE_TO_MAJOR
    )

    valid_mask = major.notna()

    n_total = adata.n_obs
    n_kept = int(
        valid_mask.sum()
    )
    n_removed = int(
        (~valid_mask).sum()
    )

    removed_summary = (
        original.loc[
            ~valid_mask
        ]
        .fillna("<missing>")
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "original_annotation"
        )
        .reset_index(
            name="n_cells"
        )
    )

    removed_summary.to_csv(
        OUTPUT_DIR
        / "removed_unmapped_cell_annotations.csv",
        index=False,
    )

    if n_removed > 0:
        print()
        print(
            "Removing cells with missing or "
            "unmapped annotations:"
        )
        print(
            removed_summary.to_string(
                index=False
            )
        )

    adata = adata[
        valid_mask.to_numpy(),
        :,
    ].copy()

    major = (
        major.loc[
            valid_mask
        ]
        .astype(str)
    )

    observed = set(
        major.unique()
    )

    final_order = [
        celltype
        for celltype in MAJOR_CELLTYPE_ORDER
        if celltype in observed
    ]

    adata.obs[
        PLOT_CELLTYPE_COLUMN
    ] = pd.Categorical(
        major.to_numpy(),
        categories=final_order,
        ordered=True,
    )

    print()
    print("=" * 72)
    print("Major-cell-type mapping")
    print("=" * 72)
    print(
        f"Original cells: "
        f"{n_total:,}"
    )
    print(
        f"Retained cells: "
        f"{n_kept:,}"
    )
    print(
        f"Removed cells:  "
        f"{n_removed:,}"
    )

    print()
    print("Major cell-type counts:")
    print(
        adata.obs[
            PLOT_CELLTYPE_COLUMN
        ]
        .value_counts(
            sort=False
        )
        .to_string()
    )

    return adata


# ============================================================
# 10. Match ranked genes to RNA object
# ============================================================
def match_ranked_genes(
    adata: ad.AnnData,
    rank_df: pd.DataFrame,
):
    """
    Match retained ranked genes to AnnData.

    Continue down the ranking until TOP_N present genes are found.
    """
    base_to_features = {}

    for feature in adata.var_names:
        feature = str(
            feature
        )

        base_symbol = re.sub(
            r"-\d+$",
            "",
            feature,
        )

        base_to_features.setdefault(
            base_symbol,
            [],
        ).append(
            feature
        )

    records = []

    for _, row in rank_df.iterrows():
        gene_symbol = row[
            "gene_symbol"
        ]

        matched_features = (
            base_to_features.get(
                gene_symbol,
                [],
            )
        )

        records.append(
            {
                "original_rank": int(
                    row["original_rank"]
                ),
                "filtered_rank": int(
                    row["filtered_rank"]
                ),
                "gene_symbol": gene_symbol,
                "score": float(
                    row["score"]
                ),
                "gene_id": row[
                    "gene_id"
                ],
                "present_in_adata": bool(
                    matched_features
                ),
                "n_features": len(
                    matched_features
                ),
                "adata_features": ";".join(
                    matched_features
                ),
            }
        )

    mapping_df = pd.DataFrame(
        records
    )

    mapping_df.to_csv(
        OUTPUT_DIR
        / "ranked_gene_mapping_to_rna.csv",
        index=False,
    )

    selected = (
        mapping_df[
            mapping_df[
                "present_in_adata"
            ]
        ]
        .sort_values(
            "filtered_rank"
        )
        .head(
            TOP_N
        )
        .copy()
    )

    if selected.empty:
        raise ValueError(
            "None of the retained ranked genes were found "
            "in the RNA AnnData object."
        )

    # Continuous rank among actually plotted genes
    selected["plot_rank"] = (
        np.arange(len(selected)) + 1
    )

    selected.to_csv(
        OUTPUT_DIR
        / "genes_used_in_dotplot.csv",
        index=False,
    )

    print()
    print("=" * 72)
    print("Genes selected for the final dot plot")
    print("=" * 72)

    print(
        selected[
            [
                "plot_rank",
                "original_rank",
                "filtered_rank",
                "gene_symbol",
                "score",
            ]
        ].to_string(
            index=False
        )
    )

    return (
        selected,
        base_to_features,
    )


# ============================================================
# 11. Build compact selected-gene AnnData
# ============================================================
def build_selected_gene_adata(
    adata: ad.AnnData,
    selected: pd.DataFrame,
    base_to_features: dict,
) -> ad.AnnData:
    """
    Build a compact AnnData containing the selected ranked genes.

    Duplicate features with the same gene symbol are summed.
    """
    gene_order = (
        selected[
            "gene_symbol"
        ]
        .tolist()
    )

    expression_columns = []
    count_columns = []

    if "counts" in adata.layers:
        count_source = (
            adata.layers["counts"]
        )
    else:
        warnings.warn(
            "layers['counts'] was not found. "
            "Detection fractions will use adata.X."
        )
        count_source = adata.X

    for gene_symbol in gene_order:
        feature_names = (
            base_to_features[
                gene_symbol
            ]
        )

        feature_indices = [
            int(
                adata.var_names.get_loc(
                    feature_name
                )
            )
            for feature_name in feature_names
        ]

        expression_column = combine_columns(
            adata.X,
            feature_indices,
        )

        count_column = combine_columns(
            count_source,
            feature_indices,
        )

        expression_columns.append(
            expression_column
        )

        count_columns.append(
            count_column
        )

    if any(
        sp.issparse(column)
        for column in expression_columns
    ):
        expression_columns = [
            column
            if sp.issparse(column)
            else sp.csr_matrix(column)
            for column in expression_columns
        ]

        expression_matrix = sp.hstack(
            expression_columns,
            format="csr",
        )

    else:
        expression_matrix = np.hstack(
            expression_columns
        )

    if any(
        sp.issparse(column)
        for column in count_columns
    ):
        count_columns = [
            column
            if sp.issparse(column)
            else sp.csr_matrix(column)
            for column in count_columns
        ]

        count_matrix = sp.hstack(
            count_columns,
            format="csr",
        )

    else:
        count_matrix = np.hstack(
            count_columns
        )

    gene_var = (
        selected
        .set_index(
            "gene_symbol"
        )
        .loc[
            gene_order,
            [
                "plot_rank",
                "original_rank",
                "filtered_rank",
                "score",
                "gene_id",
            ],
        ]
        .copy()
    )

    selected_adata = ad.AnnData(
        X=expression_matrix,
        obs=adata.obs.copy(),
        var=gene_var,
    )

    selected_adata.layers[
        "counts"
    ] = count_matrix

    selected_adata.uns[
        "log1p"
    ] = {
        "base": None
    }

    return selected_adata


# ============================================================
# 12. Calculate expression summary
# ============================================================
def calculate_expression_summary(
    selected_adata: ad.AnnData,
) -> pd.DataFrame:
    """
    Calculate mean expression and detection fraction
    for each gene and major cell type.
    """
    records = []

    categories = (
        selected_adata.obs[
            PLOT_CELLTYPE_COLUMN
        ]
        .cat.categories
        .tolist()
    )

    for cell_type in categories:
        cell_mask = (
            selected_adata.obs[
                PLOT_CELLTYPE_COLUMN
            ]
            .astype(str)
            .eq(
                cell_type
            )
            .to_numpy()
        )

        n_cells = int(
            cell_mask.sum()
        )

        if n_cells == 0:
            continue

        expression_sub = (
            selected_adata.X[
                cell_mask,
                :,
            ]
        )

        counts_sub = (
            selected_adata.layers[
                "counts"
            ][
                cell_mask,
                :,
            ]
        )

        mean_expression = mean_axis0(
            expression_sub
        )

        fraction_expressed = (
            fraction_positive_axis0(
                counts_sub
            )
        )

        for gene_index, gene_symbol in enumerate(
            selected_adata.var_names
        ):
            records.append(
                {
                    "cell_type": cell_type,
                    "gene_symbol": gene_symbol,

                    "plot_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "plot_rank",
                        ]
                    ),

                    "original_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "original_rank",
                        ]
                    ),

                    "filtered_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "filtered_rank",
                        ]
                    ),

                    "score": float(
                        selected_adata.var.loc[
                            gene_symbol,
                            "score",
                        ]
                    ),

                    "gene_id": (
                        selected_adata.var.loc[
                            gene_symbol,
                            "gene_id",
                        ]
                    ),

                    "n_cells": n_cells,

                    "mean_expression": float(
                        mean_expression[
                            gene_index
                        ]
                    ),

                    "fraction_expressed": float(
                        fraction_expressed[
                            gene_index
                        ]
                    ),
                }
            )

    summary = pd.DataFrame(
        records
    )

    summary.to_csv(
        OUTPUT_DIR
        / "ranked_gene_expression_by_major_celltype.csv",
        index=False,
    )

    return summary


# ============================================================
# 13. Nature-style dot plot
# ============================================================
def plot_dotplot(
    summary: pd.DataFrame,
    selected: pd.DataFrame,
) -> None:
    """
    Plot major-cell-type expression dot plot.

    Dot size:
        fraction expressed

    Dot color:
        mean normalized expression
    """
    gene_order = (
        selected
        .sort_values(
            "plot_rank"
        )[
            "gene_symbol"
        ]
        .tolist()
    )

    observed_celltypes = set(
        summary[
            "cell_type"
        ]
    )

    cell_order = [
        cell_type
        for cell_type in MAJOR_CELLTYPE_ORDER
        if cell_type in observed_celltypes
    ]

    if REMOVE_ZERO_EXPRESSION_GENES:
        expressed_genes = set(
            summary.loc[
                summary[
                    "fraction_expressed"
                ] > 0,
                "gene_symbol",
            ]
        )

        gene_order = [
            gene
            for gene in gene_order
            if gene in expressed_genes
        ]

    if not gene_order:
        raise ValueError(
            "No genes remain after zero-expression filtering."
        )

    plotting = summary[
        summary[
            "gene_symbol"
        ].isin(
            gene_order
        )
        & summary[
            "cell_type"
        ].isin(
            cell_order
        )
    ].copy()

    gene_to_y = {
        gene: index
        for index, gene in enumerate(
            gene_order
        )
    }

    cell_to_x = {
        cell_type: index
        for index, cell_type in enumerate(
            cell_order
        )
    }

    plotting["x"] = (
        plotting[
            "cell_type"
        ].map(
            cell_to_x
        )
    )

    plotting["y"] = (
        plotting[
            "gene_symbol"
        ].map(
            gene_to_y
        )
    )

    plotting["dot_size"] = (
        MIN_DOT_SIZE
        + plotting[
            "fraction_expressed"
        ]
        * (
            MAX_DOT_SIZE
            - MIN_DOT_SIZE
        )
    )

    color_max = np.nanpercentile(
        plotting[
            "mean_expression"
        ],
        COLOR_CLIP_PERCENTILE,
    )

    if (
        not np.isfinite(color_max)
        or color_max <= 0
    ):
        color_max = 1.0

    # --------------------------------------------------------
    # Figure geometry
    # --------------------------------------------------------
    figure_width = max(
        8.0,
        0.82 * len(cell_order) + 3.0,
    )

    figure_height = max(
        4.8,
        0.48 * len(gene_order) + 1.5,
    )

    fig = plt.figure(
        figsize=(
            figure_width,
            figure_height,
        )
    )

    # Main axes: use most of the canvas
    ax = fig.add_axes(
        [
            0.18,
            0.18,
            0.64,
            0.70,
        ]
    )

    scatter = ax.scatter(
        plotting["x"],
        plotting["y"],
        s=plotting[
            "dot_size"
        ],
        c=plotting[
            "mean_expression"
        ],
        cmap=NATURE_CMAP,
        vmin=0,
        vmax=color_max,
        edgecolors="#4C4C4C",
        linewidths=0.55,
        zorder=3,
    )

    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------
    ax.set_xticks(
        range(
            len(cell_order)
        )
    )

    ax.set_xticklabels(
        cell_order,
        rotation=42,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticks(
        range(
            len(gene_order)
        )
    )

    score_lookup = (
        selected
        .set_index(
            "gene_symbol"
        )[
            "score"
        ]
        .to_dict()
    )

    plot_rank_lookup = (
        selected
        .set_index(
            "gene_symbol"
        )[
            "plot_rank"
        ]
        .to_dict()
    )

    y_labels = [
        (
            f"{plot_rank_lookup[gene]}. "
            f"{gene} "
            f"({score_lookup[gene]:.3f})"
        )
        for gene in gene_order
    ]

    ax.set_yticklabels(
        y_labels
    )

    # Rank 1 at the top
    ax.invert_yaxis()

    ax.set_xlabel("")
    ax.set_ylabel(
        "Ranked gene (score)",
        labelpad=8,
    )

    ax.set_title(
        (
            f"Top {len(gene_order)} "
            "B-cell-ranked genes across major PBMC cell types"
        ),
        pad=12,
        fontweight="normal",
    )

    ax.set_xlim(
        -0.65,
        len(cell_order) - 0.35,
    )

    ax.set_ylim(
        len(gene_order) - 0.35,
        -0.65,
    )

    # --------------------------------------------------------
    # Grid and spines
    # --------------------------------------------------------
    ax.grid(
        axis="both",
        color="#E7E7E7",
        linewidth=0.55,
        linestyle="-",
        zorder=0,
    )

    ax.set_axisbelow(
        True
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    ax.spines[
        "left"
    ].set_linewidth(
        0.9
    )

    ax.spines[
        "bottom"
    ].set_linewidth(
        0.9
    )

    # --------------------------------------------------------
    # Color bar
    # --------------------------------------------------------
    colorbar_ax = fig.add_axes(
        [
            0.855,
            0.23,
            0.018,
            0.46,
        ]
    )

    colorbar = fig.colorbar(
        scatter,
        cax=colorbar_ax,
    )

    colorbar.set_label(
        "Mean normalized expression",
        rotation=90,
        labelpad=8,
    )

    colorbar.outline.set_linewidth(
        0.8
    )

    # --------------------------------------------------------
    # Dot-size legend
    # --------------------------------------------------------
    legend_ax = fig.add_axes(
        [
            0.84,
            0.70,
            0.15,
            0.20,
        ]
    )

    legend_ax.axis(
        "off"
    )

    legend_ax.set_xlim(
        0,
        1,
    )

    legend_ax.set_ylim(
        0,
        1,
    )

    legend_ax.text(
        0.02,
        0.97,
        "Cells expressing",
        fontsize=8,
        ha="left",
        va="top",
    )

    y_positions = np.linspace(
        0.76,
        0.14,
        len(SIZE_LEGEND_FRACTIONS),
    )

    for fraction, y_position in zip(
        SIZE_LEGEND_FRACTIONS,
        y_positions,
    ):
        dot_size = (
            MIN_DOT_SIZE
            + fraction
            * (
                MAX_DOT_SIZE
                - MIN_DOT_SIZE
            )
        )

        legend_ax.scatter(
            [0.24],
            [y_position],
            s=dot_size,
            facecolor="#D1D4D5",
            edgecolor="#4C4C4C",
            linewidth=0.55,
        )

        legend_ax.text(
            0.50,
            y_position,
            f"{fraction * 100:.0f}%",
            fontsize=8,
            va="center",
            ha="left",
        )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    output_base = (
        OUTPUT_DIR
        / OUTPUT_PREFIX
    )

    fig.savefig(
        output_base.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        output_base.with_suffix(
            ".svg"
        ),
        bbox_inches="tight",
    )

    fig.savefig(
        output_base.with_suffix(
            ".png"
        ),
        dpi=600,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    print()
    print("=" * 72)
    print("Dot plot saved")
    print("=" * 72)
    print(
        output_base.with_suffix(
            ".pdf"
        )
    )
    print(
        output_base.with_suffix(
            ".svg"
        )
    )
    print(
        output_base.with_suffix(
            ".png"
        )
    )


# ============================================================
# 14. Main
# ============================================================
def main() -> None:
    require_file(
        GENE_RANK_FILE
    )

    require_file(
        RNA_H5AD_FILE
    )

    rank_df = load_gene_rank(
        GENE_RANK_FILE
    )

    print()
    print("=" * 72)
    print("Loading RNA AnnData")
    print("=" * 72)
    print(
        RNA_H5AD_FILE
    )

    adata = sc.read_h5ad(
        RNA_H5AD_FILE
    )

    print()
    print(
        adata
    )

    adata = prepare_gene_symbols(
        adata
    )

    adata = prepare_celltypes(
        adata
    )

    selected, base_to_features = (
        match_ranked_genes(
            adata=adata,
            rank_df=rank_df,
        )
    )

    selected_adata = (
        build_selected_gene_adata(
            adata=adata,
            selected=selected,
            base_to_features=base_to_features,
        )
    )

    summary = calculate_expression_summary(
        selected_adata
    )

    plot_dotplot(
        summary=summary,
        selected=selected,
    )

    print()
    print("=" * 72)
    print("Completed")
    print("=" * 72)
    print(
        f"Output directory: "
        f"{OUTPUT_DIR}"
    )

    print()
    print("Main outputs:")
    print(
        f"  {OUTPUT_PREFIX}.pdf"
    )
    print(
        f"  {OUTPUT_PREFIX}.svg"
    )
    print(
        f"  {OUTPUT_PREFIX}.png"
    )
    print(
        "  genes_used_in_dotplot.csv"
    )
    print(
        "  filtered_ranked_genes.csv"
    )
    print(
        "  retained_ranked_genes.csv"
    )
    print(
        "  ranked_gene_mapping_to_rna.csv"
    )
    print(
        "  ranked_gene_expression_by_major_celltype.csv"
    )
    print(
        "  removed_unmapped_cell_annotations.csv"
    )


if __name__ == "__main__":
    try:
        main()

    except Exception as exc:
        print(
            f"\nERROR: {exc}",
            file=sys.stderr,
        )
        raise